## Model 3: Combined Text + Metadata (TF-IDF + Logistic Regression)

Two separate models — one trained on US bills, one on Canadian bills — evaluated both in-distribution and across countries.
Input: `text_clean_nc` (TF-IDF) concatenated with structural metadata features.  
Target: `passed` (1 = passed, 0 = failed).

In [1]:
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')

ROOT = Path("..").resolve()
NORM = ROOT / "content" / "drive" / "MyDrive" / "UBC" / "2025W" / "Term2" / "CPSC440"

Mounted at /content/drive


In [18]:
import numpy as np
import pandas as pd

from tensorflow import keras
from tensorflow.keras import layers, regularizers
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.callbacks import EarlyStopping

from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    f1_score,
    precision_score,
    recall_score,
)

### Load & Inspect Data

In [3]:
df = pd.read_csv(NORM / "bills_clean.csv")

print("Shape:", df.shape)
print("\nColumns and dtypes:")
print(df.dtypes.to_string())
print("\nNull counts:")
print(df.isnull().sum().to_string())

/tmp/ipykernel_2449/2285720059.py:1: DtypeWarning: Columns (2) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(NORM / "bills_clean.csv")


Shape: (120339, 38)

Columns and dtypes:
bill_id                          object
source                           object
session                          object
bill_number                      object
title                            object
description                      object
bill_type                        object
bill_type_raw                    object
chamber                          object
sponsor                          object
party                            object
introduced_date                  object
status                           object
passed                            int64
year                            float64
title_word_count                  int64
description_word_count            int64
month_introduced                float64
parliament_number               float64
session_number                  float64
reinstated                      float64
reached_house_second_reading    float64
reached_house_third_reading     float64
reached_senate_third_reading    float64

### Configuration

Adjust `META_BINARY_COLS`, `META_NUMERIC_COLS`, and `META_CATEGORICAL_COLS` to match the actual column names printed above.

In [12]:
TEXT_COL    = "text_clean_nc"
LABEL_COL   = "passed"
COUNTRY_COL = "source"      # values: 'us' / 'canada'
YEAR_COL    = "year"

# Temporal cutoffs — must match lstm.ipynb for a fair comparison
US_CUTOFF = 2022
CA_CUTOFF = 2020

# C-LSTM text encoding settings — must match lstm.ipynb
MAX_FEATURES = 10_000
MAX_LEN      = 512
EMBED_DIM    = 128

# Training settings
EPOCHS     = 30
BATCH_SIZE = 32

# --- Metadata column groups ---
# Only structural/relational features that have a meaningful counterpart in
# both countries. Party name (Democrat/Republican) is deliberately excluded
# because it has no cross-national equivalent and would leak country identity.
META_BINARY_COLS = [
    "is_majority_party",   # sponsor belongs to governing/majority party
    "is_government_bill",  # government bill vs private member's bill
    "chamber",             # 1 = upper (Senate/Senate), 0 = lower (House/HoC)
]

# Numeric columns to standardize
META_NUMERIC_COLS = [
    "cosponsor_count",
]

# Low-cardinality categoricals to one-hot encode
# e.g. bill_type — add/remove after inspecting df.dtypes above
META_CATEGORICAL_COLS = [
    # "bill_type",
]

### Temporal Split

In [6]:
df[TEXT_COL]  = df[TEXT_COL].astype(str)
df[LABEL_COL] = df[LABEL_COL].astype(int)
df = df.dropna(subset=[YEAR_COL]).copy()
df[YEAR_COL]  = df[YEAR_COL].astype(int)

us = df[df[COUNTRY_COL] == "us"].copy()
ca = df[df[COUNTRY_COL] == "canada"].copy()

us_train = us[us[YEAR_COL] <= US_CUTOFF]
us_test  = us[us[YEAR_COL] >  US_CUTOFF]

ca_train = ca[ca[YEAR_COL] <= CA_CUTOFF]
ca_test  = ca[ca[YEAR_COL] >  CA_CUTOFF]

print(f"US  — train: {len(us_train):,} (≤{US_CUTOFF}), test: {len(us_test):,} ({US_CUTOFF+1}–present)")
print(f"      pass rate  train: {us_train[LABEL_COL].mean():.3f}  test: {us_test[LABEL_COL].mean():.3f}")
print(f"\nCA  — train: {len(ca_train):,} (≤{CA_CUTOFF}), test: {len(ca_test):,} ({CA_CUTOFF+1}–present)")
print(f"      pass rate  train: {ca_train[LABEL_COL].mean():.3f}  test: {ca_test[LABEL_COL].mean():.3f}")

US  — train: 93,904 (≤2022), test: 19,304 (2023–present)
      pass rate  train: 0.024  test: 0.008

CA  — train: 5,411 (≤2020), test: 667 (2021–present)
      pass rate  train: 0.175  test: 0.180


### Metadata Feature Engineering

In [17]:
def _encode_binary_col(train_s: pd.Series, test_s: pd.Series, encoders: dict, col: str):
    """
    Convert one binary column to float arrays.
    If numeric (0/1), cast directly.
    If strings (e.g. 'House'/'Senate'), fit a LabelEncoder on training values
    so the integer mapping is stable and reusable for cross-country transfer.
    """
    try:
        return (
            train_s.fillna(0).values.astype(float),
            test_s.fillna(0).values.astype(float),
        )
    except (ValueError, TypeError):
        le = LabelEncoder()
        le.fit(train_s.dropna().astype(str))
        encoders.setdefault("label_encoders", {})[col] = le

        def _apply(s):
            known = set(le.classes_)
            arr = s.fillna(le.classes_[0]).astype(str).values
            arr = np.array([v if v in known else le.classes_[0] for v in arr])
            return le.transform(arr).astype(float)

        return _apply(train_s), _apply(test_s)


def _apply_binary_col(s: pd.Series, encoders: dict, col: str) -> np.ndarray:
    """Apply stored encoding for a binary column (used in cross-country transfer)."""
    le_map = encoders.get("label_encoders", {})
    if col in le_map:
        le = le_map[col]
        known = set(le.classes_)
        arr = s.fillna(le.classes_[0]).astype(str).values
        arr = np.array([v if v in known else le.classes_[0] for v in arr])
        return le.transform(arr).astype(float)
    return s.fillna(0).values.astype(float)


def build_metadata_features(train_df: pd.DataFrame, test_df: pd.DataFrame):
    """
    Fit all encoders/scalers on train_df only, transform both splits.
    Returns (X_train_meta, X_test_meta, fitted_encoders) as dense numpy arrays.
    """
    encoders = {}
    parts_train, parts_test = [], []

    # Binary columns — numeric columns are cast directly; string-valued columns
    # (e.g. 'House'/'Senate') are label-encoded using training values only so
    # the mapping is consistent for cross-country transfer.
    present_binary = [c for c in META_BINARY_COLS if c in train_df.columns]
    for col in present_binary:
        tr, te = _encode_binary_col(train_df[col], test_df[col], encoders, col)
        parts_train.append(tr.reshape(-1, 1))
        parts_test.append(te.reshape(-1, 1))
    encoders["binary_cols"] = present_binary

    # Numeric columns — fit StandardScaler on training data only
    present_numeric = [c for c in META_NUMERIC_COLS if c in train_df.columns]
    if present_numeric:
        scaler = StandardScaler()
        parts_train.append(scaler.fit_transform(train_df[present_numeric].fillna(0)))
        parts_test.append(scaler.transform(test_df[present_numeric].fillna(0)))
        encoders["scaler"] = (scaler, present_numeric)

    # Categorical columns — fit OneHotEncoder on training data only
    # handle_unknown='ignore' maps unseen categories in the target country
    # to all-zero vectors rather than raising an error during transfer.
    present_cat = [c for c in META_CATEGORICAL_COLS if c in train_df.columns]
    if present_cat:
        ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
        parts_train.append(ohe.fit_transform(train_df[present_cat].fillna("__missing__")))
        parts_test.append(ohe.transform(test_df[present_cat].fillna("__missing__")))
        encoders["ohe"] = (ohe, present_cat)

    if not parts_train:
        raise ValueError("No metadata columns found — check META_*_COLS configuration.")

    return np.hstack(parts_train), np.hstack(parts_test), encoders


def apply_metadata_encoders(df: pd.DataFrame, encoders: dict) -> np.ndarray:
    """
    Apply SOURCE country's fitted encoders to a TARGET country's DataFrame.
    Used for cross-country transfer conditions.
    """
    parts = []

    for col in encoders.get("binary_cols", []):
        parts.append(_apply_binary_col(df[col], encoders, col).reshape(-1, 1))

    if "scaler" in encoders:
        scaler, numeric_cols = encoders["scaler"]
        parts.append(scaler.transform(df[numeric_cols].fillna(0)))

    if "ohe" in encoders:
        ohe, cat_cols = encoders["ohe"]
        parts.append(ohe.transform(df[cat_cols].fillna("__missing__")))

    if not parts:
        raise ValueError("No metadata columns found — check META_*_COLS configuration.")

    return np.hstack(parts)

### Text Tokenization

In [13]:
# Separate tokenizers per country — fit on training text only
tok_us = Tokenizer(num_words=MAX_FEATURES, oov_token="<OOV>")
tok_us.fit_on_texts(us_train[TEXT_COL])

tok_ca = Tokenizer(num_words=MAX_FEATURES, oov_token="<OOV>")
tok_ca.fit_on_texts(ca_train[TEXT_COL])


def encode_text(tokenizer: Tokenizer, texts, maxlen: int = MAX_LEN) -> np.ndarray:
    seqs = tokenizer.texts_to_sequences(texts)
    return pad_sequences(seqs, maxlen=maxlen, padding="post", truncating="post")


# In-distribution text encodings
T_us_train = encode_text(tok_us, us_train[TEXT_COL])
T_us_test  = encode_text(tok_us, us_test[TEXT_COL])
T_ca_train = encode_text(tok_ca, ca_train[TEXT_COL])
T_ca_test  = encode_text(tok_ca, ca_test[TEXT_COL])

# Cross-country: SOURCE tokenizer applied to TARGET text so embedding index
# space matches what the trained model expects.
T_ca_test_via_us = encode_text(tok_us, ca_test[TEXT_COL])
T_us_test_via_ca = encode_text(tok_ca, us_test[TEXT_COL])

print(f"T_us_train: {T_us_train.shape}")
print(f"T_ca_train: {T_ca_train.shape}")

T_us_train: (93904, 512)
T_ca_train: (5411, 512)


### Build Metadata Arrays

In [19]:
# Fit metadata encoders on training data only
M_us_train, M_us_test, enc_us = build_metadata_features(us_train, us_test)
M_ca_train, M_ca_test, enc_ca = build_metadata_features(ca_train, ca_test)

# Cross-country metadata: SOURCE encoders applied to TARGET data
M_ca_test_via_us = apply_metadata_encoders(ca_test, enc_us)
M_us_test_via_ca = apply_metadata_encoders(us_test, enc_ca)

# Labels
y_us_train = us_train[LABEL_COL].values
y_us_test  = us_test[LABEL_COL].values
y_ca_train = ca_train[LABEL_COL].values
y_ca_test  = ca_test[LABEL_COL].values

META_DIM = M_us_train.shape[1]
print(f"Metadata dimension: {META_DIM}")
print(f"US  — text: {T_us_train.shape}, meta: {M_us_train.shape}")
print(f"CA  — text: {T_ca_train.shape}, meta: {M_ca_train.shape}")

Metadata dimension: 1
US  — text: (93904, 512), meta: (93904, 1)
CA  — text: (5411, 512), meta: (5411, 1)


### Combined C-LSTM + Metadata Architecture

In [20]:
def build_combined_model(meta_dim: int, max_features: int = MAX_FEATURES,
                         embed_dim: int = EMBED_DIM, max_len: int = MAX_LEN) -> keras.Model:
    # --- Text branch (C-LSTM, identical to lstm.ipynb) ---
    text_input = keras.Input(shape=(max_len,), name="text")
    x = layers.Embedding(max_features, embed_dim, input_length=max_len)(text_input)
    x = layers.Conv1D(64, kernel_size=3, padding="same")(x)
    x = layers.LeakyReLU()(x)
    x = layers.MaxPooling1D(4)(x)
    x = layers.Conv1D(32, kernel_size=3, padding="same")(x)
    x = layers.LeakyReLU()(x)
    x = layers.MaxPooling1D(4)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.5)(x)
    x = layers.LSTM(32, kernel_regularizer=regularizers.l1_l2(l1=1e-5, l2=1e-4))(x)
    x = layers.Dropout(0.5)(x)

    # --- Metadata branch ---
    meta_input = keras.Input(shape=(meta_dim,), name="meta")
    m = layers.Dense(32)(meta_input)
    m = layers.LeakyReLU()(m)
    m = layers.Dropout(0.3)(m)

    # --- Combine and predict ---
    combined = layers.Concatenate()([x, m])
    output = layers.Dense(1, activation="sigmoid")(combined)

    model = keras.Model(inputs=[text_input, meta_input], outputs=output)
    model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
    return model


def get_class_weights(labels: np.ndarray) -> dict:
    classes = np.unique(labels)
    weights = compute_class_weight("balanced", classes=classes, y=labels)
    return dict(zip(classes.tolist(), weights.tolist()))


early_stop = EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)

# --- Train US combined model ---
print("=" * 50)
print("Training model_US (C-LSTM + metadata)")
print("=" * 50)
model_US = build_combined_model(META_DIM)
model_US.summary()
cw_us = get_class_weights(y_us_train)
model_US.fit(
    {"text": T_us_train, "meta": M_us_train}, y_us_train,
    epochs=EPOCHS, batch_size=BATCH_SIZE, validation_split=0.1,
    class_weight=cw_us, callbacks=[early_stop], verbose=1,
)

# --- Train CA combined model ---
print("\n" + "=" * 50)
print("Training model_CA (C-LSTM + metadata)")
print("=" * 50)
model_CA = build_combined_model(META_DIM)
cw_ca = get_class_weights(y_ca_train)
model_CA.fit(
    {"text": T_ca_train, "meta": M_ca_train}, y_ca_train,
    epochs=EPOCHS, batch_size=BATCH_SIZE, validation_split=0.1,
    class_weight=cw_ca, callbacks=[early_stop], verbose=1,
)

Training model_US (C-LSTM + metadata)


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ text (InputLayer)   │ (None, 512)       │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding           │ (None, 512, 128)  │  1,280,000 │ text[0][0]        │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d (Conv1D)     │ (None, 512, 64)   │     24,640 │ embedding[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ leaky_re_lu         │ (None, 512, 64)   │          0 │ conv1d[0][0]      │
│ (LeakyReLU)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling1d       │ (None, 128, 64)   │          0 │ leaky_re_lu[0][0] │
│ (MaxPooling1D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_1 (Conv1D)   │ (None, 128, 32)   │      6,176 │ max_pooling1d[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ leaky_re_lu_1       │ (None, 128, 32)   │          0 │ conv1d_1[0][0]    │
│ (LeakyReLU)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling1d_1     │ (None, 32, 32)    │          0 │ leaky_re_lu_1[0]… │
│ (MaxPooling1D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 32, 32)    │        128 │ max_pooling1d_1[… │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ meta (InputLayer)   │ (None, 1)         │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 32, 32)    │          0 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 32)        │         64 │ meta[0][0]        │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm (LSTM)         │ (None, 32)        │      8,320 │ dropout[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ leaky_re_lu_2       │ (None, 32)        │          0 │ dense[0][0]       │
│ (LeakyReLU)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 32)        │          0 │ lstm[0][0]        │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_2 (Dropout) │ (None, 32)        │          0 │ leaky_re_lu_2[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 64)        │          0 │ dropout_1[0][0],  │
│ (Concatenate)       │                   │            │ dropout_2[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 1)         │         65 │ concatenate[0][0] │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 1,319,393 (5.03 MB)

 Trainable params: 1,319,329 (5.03 MB)

 Non-trainable params: 64 (256.00 B)

Epoch 1/30
2642/2642 ━━━━━━━━━━━━━━━━━━━━ 38s 12ms/step - accuracy: 0.4224 - loss: 0.7174 - val_accuracy: 0.7043 - val_loss: 0.6144
Epoch 2/30
2642/2642 ━━━━━━━━━━━━━━━━━━━━ 34s 13ms/step - accuracy: 0.5278 - loss: 0.6496 - val_accuracy: 0.9155 - val_loss: 0.4713
Epoch 3/30
2642/2642 ━━━━━━━━━━━━━━━━━━━━ 32s 12ms/step - accuracy: 0.7769 - loss: 0.5708 - val_accuracy: 0.6559 - val_loss: 0.5873
Epoch 4/30
2642/2642 ━━━━━━━━━━━━━━━━━━━━ 40s 12ms/step - accuracy: 0.7692 - loss: 0.5176 - val_accuracy: 0.5925 - val_loss: 0.5823
Epoch 5/30
2642/2642 ━━━━━━━━━━━━━━━━━━━━ 41s 12ms/step - accuracy: 0.8107 - loss: 0.4769 - val_accuracy: 0.8205 - val_loss: 0.6343
Epoch 6/30
2642/2642 ━━━━━━━━━━━━━━━━━━━━ 30s 11ms/step - accuracy: 0.8110 - loss: 0.4520 - val_accuracy: 0.9046 - val_loss: 0.4011
Epoch 7/30
2642/2642 ━━━━━━━━━━━━━━━━━━━━ 30s 12ms/step - accuracy: 0.8288 - loss: 0.4281 - val_accuracy: 0.9189 - val_loss: 0.4126
Epoch 8/30
2642/2642 ━━━━━━━━━━━━━━━━━━━━ 31s 12ms/step - accuracy: 0.8296 -

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


153/153 ━━━━━━━━━━━━━━━━━━━━ 5s 14ms/step - accuracy: 0.7505 - loss: 0.6414 - val_accuracy: 0.6974 - val_loss: 0.6997
Epoch 2/30
153/153 ━━━━━━━━━━━━━━━━━━━━ 2s 11ms/step - accuracy: 0.8010 - loss: 0.5973 - val_accuracy: 0.7952 - val_loss: 0.6169
Epoch 3/30
153/153 ━━━━━━━━━━━━━━━━━━━━ 2s 11ms/step - accuracy: 0.8460 - loss: 0.5474 - val_accuracy: 0.7528 - val_loss: 0.6308
Epoch 4/30
153/153 ━━━━━━━━━━━━━━━━━━━━ 3s 13ms/step - accuracy: 0.8110 - loss: 0.4842 - val_accuracy: 0.6531 - val_loss: 0.7673
Epoch 5/30
153/153 ━━━━━━━━━━━━━━━━━━━━ 2s 16ms/step - accuracy: 0.8193 - loss: 0.4137 - val_accuracy: 0.6089 - val_loss: 0.6413


### Evaluation

In [21]:
def evaluate(model: keras.Model, text: np.ndarray, meta: np.ndarray,
             y_true: np.ndarray, label: str) -> dict:
    y_prob = model.predict({"text": text, "meta": meta}, verbose=0).ravel()
    y_pred = (y_prob >= 0.5).astype(int)
    return {
        "Condition": label,
        "PR-AUC":    average_precision_score(y_true, y_prob),
        "ROC-AUC":   roc_auc_score(y_true, y_prob),
        "F1":        f1_score(y_true, y_pred, zero_division=0),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall":    recall_score(y_true, y_pred, zero_division=0),
    }


def print_results_table(results: list, title: str) -> None:
    header = f"{'Condition':<25}| {'PR-AUC':>7} | {'ROC-AUC':>7} | {'F1':>6} | {'Precision':>9} | {'Recall':>6}"
    sep    = "-" * len(header)
    print(f"\n=== {title} ===")
    print(sep)
    print(header)
    print(sep)
    for r in results:
        print(
            f"{r['Condition']:<25}"
            f"| {r['PR-AUC']:>7.3f} "
            f"| {r['ROC-AUC']:>7.3f} "
            f"| {r['F1']:>6.3f} "
            f"| {r['Precision']:>9.3f} "
            f"| {r['Recall']:>6.3f}"
        )
    print(sep)


results = [
    evaluate(model_US, T_us_test,        M_us_test,        y_us_test, "US → US (in-dist) "),
    evaluate(model_CA, T_ca_test,        M_ca_test,        y_ca_test, "CA → CA (in-dist) "),
    evaluate(model_US, T_ca_test_via_us, M_ca_test_via_us, y_ca_test, "US → CA (transfer)"),
    evaluate(model_CA, T_us_test_via_ca, M_us_test_via_ca, y_us_test, "CA → US (transfer)"),
]

print_results_table(results, "Model 3: Combined C-LSTM + Metadata")


=== Model 3: Combined C-LSTM + Metadata ===
--------------------------------------------------------------------------
Condition                |  PR-AUC | ROC-AUC |     F1 | Precision | Recall
--------------------------------------------------------------------------
US → US (in-dist)        |   0.065 |   0.735 |  0.098 |     0.055 |  0.418
CA → CA (in-dist)        |   0.260 |   0.675 |  0.380 |     0.250 |  0.792
US → CA (transfer)       |   0.175 |   0.429 |  0.092 |     0.219 |  0.058
CA → US (transfer)       |   0.009 |   0.497 |  0.017 |     0.009 |  0.576
--------------------------------------------------------------------------
